<a href="https://colab.research.google.com/github/maryamyazdi/Sentiment-Analysis/blob/main/Copy_of_new4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install hazm
import hazm as hz
import re
import pickle
import os
import torch
import numpy as np
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

is_gpu_enabled = torch.cuda.is_available()
device = torch.device("cuda" if is_gpu_enabled else 'cpu')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
df = pd.read_csv('/content/drive/My Drive/train.csv', sep='\t', index_col=0)
train_data = (df.head(30000)).drop(axis=1, columns='label')

def fixup(x):
    x = x.replace('\u200c', '').replace('\xa0','').replace('\r\n',' ').replace('|',' ')
    return x

normalizer = hz.Normalizer()
def my_tokenizer(text):
  text = re.sub(r"[\{\}\؛\*\=\-\+\/\n\(\)]"," ",str(text))
  text = re.sub("[ ]+"," ",text)
  text = re.sub("\!+","!",text)
  # is ? and ؟ same?
  text = re.sub("[؟]+","؟",text)
  text = re.sub("\?+","?",text)
  text = re.sub("[.]+",".",text)
  #text = re.sub("[ر]+","ر",text)
  text = fixup(normalizer.normalize(text))
  sentences = hz.sent_tokenize(text)
  words = []
  for every_sentence in sentences:
    words += hz.word_tokenize(every_sentence)
  return words

train_data['comment'] = train_data['comment'].apply(my_tokenizer)

In [5]:
  # print statistics
num_words = train_data['comment'].apply(len)

print('Min length =', num_words.min())
print('Max length =', num_words.max())
print('Mean = {:.2f}'.format(num_words.mean()))
print('Std  = {:.2f}'.format(num_words.std()))
print('mean + 2 * sigma = {:.2f}'.format(num_words.mean() + 2.0 * num_words.std()))

Min length = 2
Max length = 224
Mean = 19.42
Std  = 16.48
mean + 2 * sigma = 52.39


In [8]:
train_data

,comment,label_id
0,"[واقعا, حیف, وقت, که, بنویسم, سرویس, دهیتون, ش...",1
1,"[قرار, بود, ۱, ساعته, برسه, ولی, نیم, ساعت, زو...",0
2,"[قیمت, این, مدل, اصلا, با, کیفیتش, سازگاری, ند...",1
3,"[عالللی, بود, همه, چه, درست, و, به, اندازه, و,...",0
4,"[شیرینی, وانیلی, فقط, یک, مدل, بود, .]",0
...,...,...
29995,"[بسیار, دیر, فرستاده_شد, ., برخرد, پیک, مناسب,...",1
29996,"[پیتزا, پپرونی, آمریکایی, خوش, مزه, و, تند, بو...",1
29997,"[اصلا, مزه, این, کیک, با, کیکی, که, من, قبلا, ...",1
29998,"[غذا, خیلی, سرد, بود, !]",1


In [9]:
max_len = 53
PAD = '<pad>'

def padding_and_trimming(tokens):
  if len(tokens) < max_len:
      num_pads = max_len - len(tokens)
      tokens = [PAD] * num_pads + tokens
  elif len(tokens) > max_len:
      tokens = tokens[:max_len]
  return tokens

In [10]:
train_data['comment'] = train_data['comment'].apply(padding_and_trimming)

In [18]:
# class Vocabulary(object):
    
#     def __init__(self, tokenizer):
#         self.tokenizer = tokenizer
#         self.word2index = {}
#         self.word2count = {}
#         self.index2word = {}
#         self.count = 0
    
#     def add_word(self, word):
#         if not word in self.word2index:
#             self.word2index[word] = self.count
#             self.word2count[word] = 1
#             self.index2word[self.count] = word
#             self.count += 1
#         else:
#             self.word2count[word] += 1
    
#     def add_sentence(self, sentence):
#         for word in self.tokenizer(sentence):
#             self.add_word(word)

In [19]:
# PAD = '<pad>'
# UNK = '<unk>'

# class TextClassificationDataset(Dataset):

#     def __init__(self, tokenizer, vocab_path, max_len , min_count):
  
#         self.vocab_path = vocab_path
#         self.tokenizer = tokenizer
#         self.max_len = max_len
#         self.min_count = min_count
        
#         self.cache = {}
#         self.vocab = None
        
#         self.classes = []
#         self.class_to_index = {}

#         self.cache = {}
#         self.vocab = None
        
#         self.classes = []
#         self.class_to_index = {}
#         self.text_files = []
        
#         split_path = f'{path}/{split}'
        
#         for cls_idx, label in enumerate(os.listdir(split_path)):
#             text_files = [(fname, cls_idx) for fname in glob(f'{split_path}/{label}/*.txt')]
#             self.text_files += text_files
#             self.classes += [label]
#             self.class_to_index[label] = cls_idx
        
#         self.num_classes = len(self.classes)
            
#         # build vocabulary from training and valida
            
#         # build vocabulary from training and validation texts
#         self.build_vocab()
        
#     def __getitem__(self, index):
#         # read the tokenized text file and its label (neg=0, pos=1)
#         # fname, class_idx = self.text_files[index]
#         # if fname in self.cache:
#           #return self.cache[fname], class_idx
#           # print("done")
#           # return

#         # read text file 
#         # text = open(fname).read()
#         text = train_data['comment'][0]

#         # tokenize the text file
#         tokens = self.tokenizer(text)
        
#         # padding and trimming
#         if len(tokens) < self.max_len:
#             num_pads = self.max_len - len(tokens)
#             tokens = [PAD] * num_pads + tokens
#         elif len(tokens) > self.max_len:
#             tokens = tokens[:self.max_len]
            
#         # numericalizing
#         ids = torch.LongTensor(self.max_len)
#         for i, word in enumerate(tokens):
#             if word not in self.vocab.word2index:
#                 ids[i] = self.vocab.word2index[UNK]  # unknown words
#             elif word != PAD and self.vocab.word2count[word] < self.min_count:
#                 ids[i] = self.vocab.word2index[UNK]  # rare words
#             else:
#                 ids[i] = self.vocab.word2index[word]
                
#         # save in cache for future use
#         # self.cache[fname] = ids
        
#         return ids
    
#     def build_vocab(self):
#         if not os.path.exists(self.vocab_path):
#             vocab = Vocabulary(self.tokenizer)
#             with open('/content/drive/My Drive/train.csv', encoding='utf8') as f:
#               for (i, line) in enumerate(f):
#                 vocab.add_sentence((line.split(sep='\t'))[1])
#                 if i > 2000:
#                   break

#             # sort words by their frequencies
#             words = [(0, PAD), (0, UNK)]
#             words += sorted([(c, w) for w, c in vocab.word2count.items()], reverse=True)

#             self.vocab = Vocabulary(self.tokenizer)
#             for i, (count, word) in enumerate(words):
#                 self.vocab.word2index[word] = i
#                 self.vocab.word2count[word] = count
#                 self.vocab.index2word[i] = word
#                 self.vocab.count += 1

#             pickle.dump(self.vocab, open(self.vocab_path, 'wb'))
#         else:
#             self.vocab = pickle.load(open(self.vocab_path, 'rb'))

In [ ]:
# min_count = 10
# vocab_path = 'vocab.pkl'
# max_len = np.mean(num_words) + 2.0 * np.std(num_words)
# train_ds = TextClassificationDataset(my_tokenizer, vocab_path, max_len, min_count)

# vocab = train_ds.vocab
# freqs = [(count, word) for (word, count) in vocab.word2count.items() if count >= min_count]
# vocab_size = len(freqs) + 2  # for PAD and UNK tokens
# print(f'Vocab size = {vocab_size}')

# print('\nMost common words:')
# for c, w in sorted(freqs, reverse=True)[:10]:
#     print(f'{w}: {c}')

In [ ]:
!wget 'http://vectors.nlpl.eu/repository/20/61.zip' -O '/content/drive/MyDrive/w2vec.zip'

In [ ]:
!unzip /content/drive/MyDrive/w2vec.zip -d /content/drive/MyDrive/w2vec

In [7]:
import gensim
w2v_model = gensim.models.KeyedVectors.load_word2vec_format('/content/drive/MyDrive/w2vec/model.txt', binary=False, unicode_errors='replace')
w2v_weights = torch.FloatTensor(w2v_model.vectors)

In [11]:
import torch.nn as nn


class LSTMClassifier(nn.Module):
  def __init__(self, hidden_size, batch_size, layers_count):
    super(LSTMClassifier, self).__init__()
    self.hidden_size = hidden_size
    self.batch_size = batch_size
    self.layers_count = layers_count

    self.embedding = nn.Embedding.from_pretrained(w2v_weights)
    self.lstm = nn.LSTM(100, hidden_size, layers_count, dropout=0.2, bidirectional=False, batch_first=True)
    self.classifier_layer = nn.Sequential(
        nn.Linear(hidden_size, 100),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(100, 2)
    )
    self.hidden = self.init_hidden()

  def init_hidden(self):
    h = torch.autograd.Variable(torch.zeros((self.layers_count, self.batch_size, self.hidden_size)).to(device))
    c = torch.autograd.Variable(torch.zeros((self.layers_count, self.batch_size, self.hidden_size)).to(device))
    return h, c

  def forward(self, x):
    x = self.embedding(x)
    x, self.hidden = self.lstm(x, self.hidden)
    x = x.permute(1, 0, 2).detach()
    x = self.classifier_layer(x[-1])
    return x


In [12]:
BATCH_SIZE = 3000
lstm_model = LSTMClassifier(hidden_size=256, batch_size=BATCH_SIZE, layers_count=1)

if is_gpu_enabled:
  lstm_model = lstm_model.cuda()

/usr/local/lib/python3.7/dist-packages/torch/nn/modules/rnn.py:65: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  "num_layers={}".format(dropout, num_layers))


In [13]:
criterion = nn.CrossEntropyLoss()
if is_gpu_enabled:
  criterion = criterion.cuda()

optimizer = torch.optim.SGD(lstm_model.parameters(), lr=0.001, momentum=0.9)
# scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=1, gamma=0.9)

In [15]:
PAD = '<pad>'
UNK = '<unk>'

class TDataset(torch.utils.data.Dataset):
    def __init__(self, dataset):
        self.X_train = dataset['comment']
        self.y_train = dataset['label_id']

    def __len__(self):
        return len(self.X_train)

    def __getitem__(self, index):
        vectors = []
        for token in self.X_train[index]:
          if token == PAD:
            vectors.append(0)
            continue
          try:
            vectors.append(w2v_model.vocab[token].index)
          except KeyError:
            vectors.append(0)
        return torch.tensor(vectors), torch.tensor(self.y_train[index])


dataset = TDataset(train_data)

In [16]:
train_dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

lstm_model.train()
torch.autograd.set_detect_anomaly(True)

losses = []
for epoch in range(5):
  total_loss = 0
  for i, (inputs, targets) in enumerate(train_dataloader):
    inputs = torch.autograd.Variable(inputs.to(device))
    targets = torch.autograd.Variable(targets.to(device))

    lstm_model.zero_grad()
    outputs = lstm_model(inputs)
    
    loss = criterion(outputs, targets)
    loss.backward(retain_graph=True)
    torch.nn.utils.clip_grad_norm_(lstm_model.parameters(), 0.3)        
    optimizer.step()
    total_loss += loss.item()
    
    print(f'Epoch {epoch + 1}/5 : step {i + 1}/{len(train_dataloader) // BATCH_SIZE}, loss: {loss.item()}')

Epoch 1/5 : step 1/0, loss: 0.693337619304657
Epoch 1/5 : step 2/0, loss: 0.6927505731582642
Epoch 1/5 : step 3/0, loss: 0.6933921575546265
Epoch 1/5 : step 4/0, loss: 0.6933314800262451
Epoch 1/5 : step 5/0, loss: 0.693483293056488
Epoch 1/5 : step 6/0, loss: 0.6935248970985413
Epoch 1/5 : step 7/0, loss: 0.6933437585830688
Epoch 1/5 : step 8/0, loss: 0.6936880946159363
Epoch 1/5 : step 9/0, loss: 0.6929287314414978


RuntimeError: ignored

In [170]:
total_loss

4.159211039543152

In [ ]:
!pip install transformers
from transformers import AutoTokenizer, AutoModelForMaskedLM

In [ ]:
tokenizer = AutoTokenizer.from_pretrained('xlm-roberta-base')
model = AutoModelForMaskedLM.from_pretrained("xlm-roberta-base")


Downloading:   0%|          | 0.00/512 [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/4.83M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/8.68M [00:00<?, ?B/s]

Downloading:   0%|          | 0.00/1.04G [00:00<?, ?B/s]